# Task 13: chosen_3 — ddg_esm2 + ddg_struct + ddg_rasp 联合

**目的**: 去掉始终拖后腿的 FoldX，将三种最有希望的 ΔΔG 联合在一起。

| ddg | 方法 | 类型 |
|---|---|---|
| `ddg_esm2` | ESM2 zero-shot masked marginal | 序列 (PLM) |
| `ddg_struct` | MJ 统计势能 (接触势 + 溶剂化) | 结构 (统计) |
| `ddg_rasp` | RaSP 3D-CNN cavity | 结构 (深度学习) |

**对比**: PCA(50) 基线 vs PCA(50) + chosen_3

In [7]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载特征矩阵 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
esm_cols = [c for c in feat_cols if c.startswith("esm_")]
STRUCT_NAMES = ["plddt","sasa","rsa","ss_helix","ss_strand","ss_coil",
                "delta_hydrophobicity"]

X_full_arr = df_feat[feat_cols].values.astype(np.float32)
esm_idx = [feat_cols.index(c) for c in esm_cols]
struct_idx = [feat_cols.index(c) for c in STRUCT_NAMES if c in feat_cols]
X_esm = X_full_arr[:, esm_idx]
X_struct = X_full_arr[:, struct_idx]

y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values
full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()

print(f"n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={y_bin.sum()/len(y_bin):.4f}")

# ===== 加载 chosen_3: ddg_esm2 + ddg_struct + ddg_rasp =====
CHOSEN = ["ddg_esm2", "ddg_struct", "ddg_rasp"]
ddg_files = {
    "ddg_esm2":   ("ddg_esm2.csv",   "ddg_esm2"),
    "ddg_struct": ("ddg_struct.csv", "ddg_struct"),
    "ddg_rasp":   ("ddg_rasp.csv",   "ddg_rasp"),
}

ddg_sources = {}
for name, (fname, dcol) in ddg_files.items():
    path = BASE_PATH + fname
    df_d = pd.read_csv(path)
    ddg_map = dict(zip(df_d["KEY"], df_d[dcol]))
    arr = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    n_valid = np.isfinite(arr).sum()
    ddg_sources[name] = arr.reshape(-1, 1)
    print(f"  {name:12s}: {n_valid}/{len(arr)} 有效")

n=2179, pos=235, base_rate=0.1078
  ddg_esm2    : 2179/2179 有效
  ddg_struct  : 2168/2179 有效
  ddg_rasp    : 2168/2179 有效


## 13a. CV: PCA(50) vs PCA(50) + chosen_3

In [8]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
N_COMP = 50

oof_pca = np.zeros(len(y_bin), dtype=np.float32)
oof_chosen3 = np.zeros(len(y_bin), dtype=np.float32)

for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full_arr, y_bin, groups)):
    y_tr = y_bin[tr_idx]; y_te = y_bin[te_idx]

    # --- ESM2 → PCA ---
    imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
    Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
    Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
    pca = PCA(n_components=N_COMP, random_state=42)
    Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
    Xe_te_pca = pca.transform(Xe_te).astype(np.float32)

    # --- Struct ---
    imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
    Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
    Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)

    X_tr_base = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
    X_te_base = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)

    sw = compute_sample_weight("balanced", y_tr)

    def fit_pred(X_tr, X_te):
        m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.5,
                          objective="binary:logistic", eval_metric="aucpr",
                          random_state=42, n_jobs=-1, tree_method="hist")
        m.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        return m.predict_proba(X_te)[:, 1]

    # ---- PCA 基线 ----
    oof_pca[te_idx] = fit_pred(X_tr_base, X_te_base)

    # ---- PCA + chosen_3 ----
    ddg_parts_tr = []; ddg_parts_te = []
    for name in CHOSEN:
        imp_d = SimpleImputer(strategy="median")
        ddg_parts_tr.append(imp_d.fit_transform(ddg_sources[name][tr_idx]).astype(np.float32))
        ddg_parts_te.append(imp_d.transform(ddg_sources[name][te_idx]).astype(np.float32))
    oof_chosen3[te_idx] = fit_pred(
        np.hstack([X_tr_base] + ddg_parts_tr).astype(np.float32),
        np.hstack([X_te_base] + ddg_parts_te).astype(np.float32))

    a_p = roc_auc_score(y_te, oof_pca[te_idx])
    a_c = roc_auc_score(y_te, oof_chosen3[te_idx])
    print(f"  Fold {fold}: PCA={a_p:.4f}  PCA+chosen_3={a_c:.4f}  Δ={a_c-a_p:+.4f}")

  Fold 0: PCA=0.6685  PCA+chosen_3=0.7043  Δ=+0.0359
  Fold 1: PCA=0.6246  PCA+chosen_3=0.6398  Δ=+0.0151
  Fold 2: PCA=0.5527  PCA+chosen_3=0.5799  Δ=+0.0273
  Fold 3: PCA=0.6048  PCA+chosen_3=0.6089  Δ=+0.0041
  Fold 4: PCA=0.6204  PCA+chosen_3=0.6161  Δ=-0.0043


## 13b. 汇总

In [9]:
br = float(y_bin.sum() / len(y_bin))
auroc_pca = roc_auc_score(y_bin, oof_pca)
auprc_pca = average_precision_score(y_bin, oof_pca)
auroc_c3  = roc_auc_score(y_bin, oof_chosen3)
auprc_c3  = average_precision_score(y_bin, oof_chosen3)

print(f"\n{'='*70}")
print(f"  Task 13: chosen_3 (ddg_esm2 + ddg_struct + ddg_rasp)")
print(f"  n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={br:.4f}")
print(f"  {'配置':<35s} {'AUROC':>8s} {'AUPRC':>8s} {'ΔAUROC':>12s}")
print(f"  {'-'*65}")
print(f"  {'PCA(50) 基线':<35s} {auroc_pca:>8.4f} {auprc_pca:>8.4f} {'baseline':>12s}")
d = auroc_c3 - auroc_pca
m = " +" if d > 0.005 else ("" if d > -0.005 else " -")
print(f"  {'PCA(50) + chosen_3':<35s} {auroc_c3:>8.4f} {auprc_c3:>8.4f} {d:>+12.4f}{m}")
print(f"{'='*70}")
print(f"\nAlphaMissense baseline: 0.6374, gap={0.6374 - auroc_c3:.4f}")


  Task 13: chosen_3 (ddg_esm2 + ddg_struct + ddg_rasp)
  n=2179, pos=235, base_rate=0.1078
  配置                                     AUROC    AUPRC       ΔAUROC
  -----------------------------------------------------------------
  PCA(50) 基线                            0.6144   0.1558     baseline
  PCA(50) + chosen_3                    0.6290   0.1704      +0.0146 +

AlphaMissense baseline: 0.6374, gap=0.0084


## 13c. 特征重要性 (PCA + chosen_3, fit on all data)

In [10]:
print("\n--- 特征重要性 PCA(50)+chosen_3 ---")

imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
Xe_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
pca_all = PCA(n_components=N_COMP, random_state=42)
Xe_pca_all = pca_all.fit_transform(Xe_all).astype(np.float32)

imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
Xs_all = scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32)

parts = [Xe_pca_all, Xs_all]
for name in CHOSEN:
    imp_d = SimpleImputer(strategy="median")
    d_all = imp_d.fit_transform(ddg_sources[name]).astype(np.float32)
    parts.append(d_all)

X_all = np.hstack(parts).astype(np.float32)
all_names = [f"PC{i+1}" for i in range(N_COMP)] + STRUCT_NAMES + CHOSEN

sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)
xgb_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.5,
                       scale_pos_weight=sw_ratio,
                       objective="binary:logistic", eval_metric="aucpr",
                       random_state=42, n_jobs=-1, tree_method="hist")
xgb_fi.fit(X_all, y_bin, verbose=False)

imps = xgb_fi.feature_importances_
idx_sorted = np.argsort(imps)[::-1]

print("Top-20 特征:")
for rank, i in enumerate(idx_sorted[:20]):
    val = imps[i]; bar = "\u2588" * int(val * 2000)
    print(f"  {rank+1:2d}. {all_names[i]:30s}  {val:.5f}  {bar}")

print(f"\nchosen_3 排名 (共 {len(all_names)} 个特征):")
for name in CHOSEN:
    i = all_names.index(name)
    rank = int(np.where(idx_sorted == i)[0][0]) + 1
    marker = " \u2605 top-5!" if rank <= 5 else (" \u2605 top-10!" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else ""))
    print(f"  {name:15s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")

print(f"\n结构特征排名:")
for name in STRUCT_NAMES:
    i = all_names.index(name)
    rank = int(np.where(idx_sorted == i)[0][0]) + 1
    marker = " \u2605 top-10!" if rank <= 10 else (" \u2605 top-20!" if rank <= 20 else "")
    print(f"  {name:25s} 排名={rank:3d}/{len(all_names)}  重要性={imps[i]:.5f}{marker}")


--- 特征重要性 PCA(50)+chosen_3 ---
Top-20 特征:
   1. ddg_esm2                        0.03966  ███████████████████████████████████████████████████████████████████████████████
   2. PC36                            0.02283  █████████████████████████████████████████████
   3. ss_coil                         0.02066  █████████████████████████████████████████
   4. plddt                           0.02049  ████████████████████████████████████████
   5. ddg_struct                      0.02011  ████████████████████████████████████████
   6. PC42                            0.01908  ██████████████████████████████████████
   7. ddg_rasp                        0.01870  █████████████████████████████████████
   8. PC8                             0.01854  █████████████████████████████████████
   9. PC33                            0.01849  ████████████████████████████████████
  10. PC1                             0.01847  ████████████████████████████████████
  11. PC11                            0.01839  █

## 13d. 三种 ddg 间相关性 + 单特征 AUROC

In [ ]:
from scipy.stats import spearmanr

ddg_mat = np.column_stack([ddg_sources[n].ravel() for n in CHOSEN])

print(f"\n{'='*70}")
print(f"  chosen_3 Spearman 秩相关矩阵")
print(f"  {'':>14s}", end="")
for n in CHOSEN:
    print(f"{n:>14s}", end="")
print()

for i, ni in enumerate(CHOSEN):
    print(f"  {ni:>14s}", end="")
    for j, nj in enumerate(CHOSEN):
        mask = np.isfinite(ddg_mat[:, i]) & np.isfinite(ddg_mat[:, j])
        r, _ = spearmanr(ddg_mat[mask, i], ddg_mat[mask, j])
        print(f"{r:>14.4f}", end="")
    print()

print(f"\n  单特征 AUROC:")
for i, name in enumerate(CHOSEN):
    vals = ddg_mat[:, i]
    mask = np.isfinite(vals)
    if y_bin[mask].sum() > 0 and (y_bin[mask] == 0).sum() > 0:
        auc_single = roc_auc_score(y_bin[mask], vals[mask])
        print(f"    {name:15s}  AUROC={auc_single:.4f}  (n={mask.sum()})")
print(f"{'='*70}")